In [1]:
!pip install transformers datasets torch torchvision

In [2]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from tqdm import tqdm


In [3]:
df = pd.read_csv("/content/dataset.csv")

print("Head:")
display(df.head())

print("\nCategory counts:")
print(df['category'].value_counts())


Head:


,text,category
0,mcdonalds combo meal burger fries takeaway,food
1,kfc chicken bucket dinner fast food,food
2,dominos pizza pepperoni order,food
3,subway sandwich lunch meal,food
4,starbucks coffee latte cappuccino,food



Category counts:
category
food          23
groceries     23
travel        23
shopping      23
utilities     23
healthcare    23
misc          23
Name: count, dtype: int64


In [4]:
label_encoder = LabelEncoder()
df['label_id'] = label_encoder.fit_transform(df['category'])

label2id = {c: int(i) for c, i in zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))}
id2label = {i: c for c, i in enumerate(label_encoder.classes_)}

label2id, id2label


({'food': 0,
  'groceries': 1,
  'healthcare': 2,
  'misc': 3,
  'shopping': 4,
  'travel': 5,
  'utilities': 6},
 {'food': 0,
  'groceries': 1,
  'healthcare': 2,
  'misc': 3,
  'shopping': 4,
  'travel': 5,
  'utilities': 6})

In [5]:
train_df, val_df = train_test_split(df, test_size=0.15, random_state=42, stratify=df['label_id'])

print(len(train_df), len(val_df))


136 25


In [6]:
class ExpenseDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len=32):
        """
        texts: list/series of raw text strings
        labels: list/series of integer category labels
        tokenizer: DistilBERT tokenizer (pretrained)
        max_len: sequence length for padding/truncation
        """
        self.texts = df['text'].tolist()
        self.labels = df['label_id'].tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len


    def __len__(self):
        """Return total number of examples."""
        return len(self.texts)


    def __getitem__(self, idx):
        """
        Return one encoded example:
        - input_ids
        - attention_mask
        - label
        """
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )
        return {
            'input_ids': enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'labels': torch.tensor(self.labels[idx])
        }

In [7]:
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(label2id),
    label2id=label2id,
    id2label=id2label
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [8]:
train_dataset = ExpenseDataset(train_df, tokenizer)
val_dataset = ExpenseDataset(val_df, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)


In [9]:
def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for batch in loader:
        optimizer.zero_grad()
        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = model(**batch)
        loss = outputs.loss
        logits = outputs.logits

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = logits.argmax(dim=1)
        correct += (preds == batch['labels']).sum().item()
        total += len(preds)

    return total_loss / len(loader), correct / total


def evaluate(model, loader, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0

    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}

            outputs = model(**batch)
            loss = outputs.loss
            logits = outputs.logits

            total_loss += loss.item()
            preds = logits.argmax(dim=1)
            correct += (preds == batch['labels']).sum().item()
            total += len(preds)

    return total_loss / len(loader), correct / total


In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

EPOCHS = 4

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print("-"*30)

    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, device)

    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")



Epoch 1/4
------------------------------
Train Loss: 1.8918, Train Acc: 0.2112
Val Loss: 1.7353, Val Acc: 0.7267

Epoch 2/4
------------------------------
Train Loss: 1.5900, Train Acc: 0.7640
Val Loss: 1.2937, Val Acc: 0.9255

Epoch 3/4
------------------------------
Train Loss: 1.1981, Train Acc: 0.9193
Val Loss: 0.8407, Val Acc: 0.9814

Epoch 4/4
------------------------------
Train Loss: 0.7236, Train Acc: 0.9938
Val Loss: 0.5140, Val Acc: 1.0000


In [11]:
save_path = "/content/expense_classifier_model"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

import json
with open(save_path + "/label_map.json", "w") as f:
    json.dump(id2label, f)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [15]:
import json

path = "/content/expense_classifier_model/label_map.json"

# Load your CURRENT (wrong-direction) map
with open(path, "r") as f:
    raw = json.load(f)

# Reverse it → make keys the IDs ("0","1","2"...)
id2label = {str(v): k for k, v in raw.items()}
label2id = {k: int(v) for v, k in id2label.items()}

print("Corrected id2label:", id2label)
print("Corrected label2id:", label2id)

# Overwrite the file with corrected mapping
with open(path, "w") as f:
    json.dump(id2label, f, indent=4)


Corrected id2label: {'0': 'food', '1': 'groceries', '2': 'healthcare', '3': 'misc', '4': 'shopping', '5': 'travel', '6': 'utilities'}
Corrected label2id: {'food': 0, 'groceries': 1, 'healthcare': 2, 'misc': 3, 'shopping': 4, 'travel': 5, 'utilities': 6}


In [17]:
import json

with open("/content/expense_classifier_model/config.json") as f:
    cfg = json.load(f)

print(cfg["id2label"])
print(cfg["label2id"])


{'food': 0, 'groceries': 1, 'healthcare': 2, 'misc': 3, 'shopping': 4, 'travel': 5, 'utilities': 6}
{'food': 0, 'groceries': 1, 'healthcare': 2, 'misc': 3, 'shopping': 4, 'travel': 5, 'utilities': 6}


In [18]:
correct_id2label = {
    "0": "food",
    "1": "groceries",
    "2": "healthcare",
    "3": "misc",
    "4": "shopping",
    "5": "travel",
    "6": "utilities"
}

correct_label2id = {
    "food": 0,
    "groceries": 1,
    "healthcare": 2,
    "misc": 3,
    "shopping": 4,
    "travel": 5,
    "utilities": 6
}
cfg["id2label"] = {int(k): v for k, v in correct_id2label.items()}
cfg["label2id"] = correct_label2id
cfg["num_labels"] = 7


In [19]:
with open("/content/expense_classifier_model/config.json", "w") as f:
    json.dump(cfg, f, indent=4)


In [20]:
model_path = "/content/expense_classifier_model"
tokenizer = DistilBertTokenizerFast.from_pretrained(model_path)
model = DistilBertForSequenceClassification.from_pretrained(
    model_path
)
model.eval()


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [21]:
def predict(text):
    enc = tokenizer(text, return_tensors="pt", truncation=True, padding="max_length", max_length=32)
    with torch.no_grad():
        logits = model(**enc).logits
        probs = torch.softmax(logits, dim=1)
        pred = torch.argmax(probs).item()
    return cfg["id2label"][pred], float(probs[0][pred])

print(predict("mcdonalds burger"))
print(predict("amazon electronics"))
print(predict("petrol indian oil"))
print(predict("hospital medical test"))
print(predict("milk bread grocery"))


('food', 0.7494964599609375)
('shopping', 0.5359472036361694)
('travel', 0.5051593780517578)
('healthcare', 0.7183131575584412)
('groceries', 0.5082412958145142)
